# FIFA World Cup — Modeling with XGBoost & 2026 Predictions

Extended version of the modeling notebook that adds **XGBoost** and **LightGBM**
to the model comparison alongside the existing baselines, Logistic Regression,
Random Forest, and HistGradientBoosting.

Covers **Steps 3–7**: target definition, the leakage-free feature set,
all models (including gradient boosting), time-based validation, hyperparameter
sensitivity analysis, and the 2026 forecast.

> **Prerequisite:** `libomp` must be installed (`brew install libomp` on macOS)
> for XGBoost and LightGBM to work.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import (config, data, evaluate, features, poisson,  # noqa: E402
                 predict, train)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss as sk_log_loss

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
FIG = config.FIGURES_DIR

print("XGBoost and LightGBM imported successfully")

XGBoost and LightGBM imported successfully


## 3. Target definition & 4. The modeling dataset

**Target** (3-class, from `team_a`'s perspective): `team_a_win` / `draw` /
`team_b_win`, derived from the **regulation+ET goal score** (penalty shoot-outs
count as draws — see EDA).

**Neutral teams.** Because venues are neutral and the raw home/away label is
biased, each match is emitted **once** with `team_a`/`team_b` assigned by a
seeded coin flip, and the model consumes **antisymmetric difference features**
(`team_a − team_b`). Swapping the two teams flips every feature's sign and flips
the label — exactly the symmetry a neutral-venue model should respect.

**Leakage prevention.** Every feature is computed in a single chronological pass
that records each team's state *before* applying the match result, so a match's
features only ever see strictly-earlier matches.

In [2]:
matches = data.load_men_matches()
df = features.build_modeling_dataset(matches)

print("Modeling dataset:", df.shape)
print("\nTarget distribution (balanced after symmetrisation):")
print(df["target"].map(config.INT_TO_OUTCOME).value_counts(normalize=True).round(3))
print("\nFeatures used:")
print(features.FEATURE_COLUMNS)

df[["year", "team_a_name", "team_b_name", "elo_diff", "form_points_diff",
    "host_advantage", "h2h_played", "goals_a", "goals_b", "target"]].head()

Modeling dataset: (960, 34)

Target distribution (balanced after symmetrisation):
target
team_b_win    0.394
team_a_win    0.388
draw          0.219
Name: proportion, dtype: float64

Features used:
['elo_diff', 'matches_played_diff', 'win_rate_diff', 'gf_avg_diff', 'ga_avg_diff', 'form_points_diff', 'experience_diff', 'host_advantage', 'h2h_played', 'h2h_a_win_rate', 'h2h_goal_diff_avg', 'same_confederation', 'knockout_stage']


,year,team_a_name,team_b_name,elo_diff,form_points_diff,host_advantage,h2h_played,goals_a,goals_b,target
0,1930,France,Mexico,0.0,NaN,0,0,4,1,0
1,1930,Belgium,United States,0.0,NaN,0,0,0,3,2
2,1930,Yugoslavia,Brazil,0.0,NaN,0,0,2,1,0
3,1930,Romania,Peru,0.0,NaN,0,0,3,1,0
4,1930,France,Argentina,35.0,NaN,0,0,0,1,2


## 5 & 6. Models and time-based validation

**Why not a random split?** Features (Elo, cumulative stats) are built from the
past, and a random split would leak future tournaments through the shared
rating state. We instead run a **temporal backtest**: to predict tournament `Y`,
train only on matches from years `< Y`. We evaluate the last four World Cups
(2010–2022).

**Metrics.** Accuracy is intuitive but misleading (draws are rarely the modal
class). The honest, *primary* metric is **log loss** — a proper scoring rule
that rewards calibrated probabilities — complemented by the **Brier score**.
Macro-F1 checks the draw class isn't ignored.

**Models in this notebook:** Majority baseline, Prior baseline, Logistic
Regression, Random Forest, HistGradientBoosting, **XGBoost**, and **LightGBM**.

In [3]:
comparison = evaluate.compare_models(df)
print(f"Models compared: {list(comparison['model'])}")
comparison

Models compared: ['random_forest', 'logistic_regression', 'xgboost', 'lightgbm', 'hist_gradient_boosting', 'prior_baseline', 'majority_baseline']


,model,n,accuracy,macro_f1,log_loss,brier
0,random_forest,256,0.554688,0.416931,1.005776,0.597712
1,logistic_regression,256,0.496094,0.388929,1.040500,0.613913
2,xgboost,256,0.476562,0.423672,1.115755,0.655918
3,lightgbm,256,0.453125,0.399911,1.135210,0.666088
4,hist_gradient_boosting,256,0.457031,0.412149,1.217051,0.694897
5,prior_baseline,256,0.398438,0.356611,21.682510,1.203125
6,majority_baseline,256,0.382812,0.184557,22.245692,1.234375


The two **baselines** (majority class, prior probabilities) have catastrophic
log loss — they are uncalibrated. The real models all beat them comfortably.

With XGBoost and LightGBM now included, we can see how gradient-boosting
methods compare to simpler approaches on this small (~960-match) dataset.
Gradient boosters sometimes overfit on small data, but they can also capture
non-linear feature interactions that logistic regression misses.

We pick the best non-baseline model by log loss and inspect it per tournament.

In [4]:
best = str(comparison[~comparison["model"].str.contains("baseline")].iloc[0]["model"])
print("Best model by log loss:", best)

per_year, overall = evaluate.temporal_backtest(
    df, lambda: train.build_models()[best])
display(per_year)

# pooled confusion matrix across the backtest years
pt, pp = [], []
for ty in config.BACKTEST_YEARS:
    tr, te = df[df["year"] < ty], df[df["year"] == ty]
    if len(tr) and len(te):
        mdl = train.fit_model(train.build_models()[best], tr)
        pp.append(train.predict_proba(mdl, te)); pt.append(te["target"].to_numpy())
cm = evaluate.confusion(np.concatenate(pt), np.concatenate(pp))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion matrix — {best} (pooled backtest)")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.show()

Best model by log loss: random_forest


,year,accuracy,macro_f1,log_loss,brier,n
0,2010,0.515625,0.396104,1.037819,0.624403,64
1,2014,0.562500,0.417527,0.997119,0.592759,64
2,2018,0.578125,0.429293,0.993896,0.589447,64
3,2022,0.562500,0.425078,0.994267,0.584239,64


/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/90224172.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.show()


The confusion matrix shows the expected behaviour: the model **almost never
predicts a draw as the most-likely class** (draws are genuinely hard and never
the modal outcome of an individual neutral match). It still assigns ~20–30%
*probability* to draws, which is why probabilistic metrics are the right lens.

### Feature importance / coefficients

In [5]:
# For Random Forest, show the Gini importances
rf = train.fit_model(train.build_models()[best], df)
if hasattr(rf, "feature_importances_"):
    imp = pd.Series(rf.feature_importances_, index=features.FEATURE_COLUMNS).sort_values()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    imp.plot(kind="barh", ax=ax, color="#4C72B0")
    ax.set_title(f"Feature importances — {best}")
    plt.tight_layout(); plt.savefig(FIG / "feature_importance.png", dpi=110); plt.show()

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/2919967150.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "feature_importance.png", dpi=110); plt.show()


### XGBoost deep dive

Now that XGBoost is available, let's inspect it more closely:
per-tournament performance, feature importance (gain-based vs. the Random
Forest's Gini importance), and a quick hyperparameter sensitivity check.

In [6]:
# XGBoost per-tournament backtest
xgb_per_year, xgb_overall = evaluate.temporal_backtest(
    df, lambda: train.build_models()["xgboost"])
print("XGBoost per-tournament:")
display(xgb_per_year)
print("\nXGBoost pooled:", xgb_overall.to_dict("records"))

# LightGBM per-tournament backtest
lgbm_per_year, lgbm_overall = evaluate.temporal_backtest(
    df, lambda: train.build_models()["lightgbm"])
print("\nLightGBM per-tournament:")
display(lgbm_per_year)
print("\nLightGBM pooled:", lgbm_overall.to_dict("records"))

XGBoost per-tournament:


,year,accuracy,macro_f1,log_loss,brier,n
0,2010,0.43750,0.422808,1.213354,0.716616,64
1,2014,0.50000,0.434126,1.098224,0.644323,64
2,2018,0.46875,0.367251,1.096980,0.649431,64
3,2022,0.50000,0.437546,1.054464,0.613300,64



XGBoost pooled: [{'scope': 'pooled', 'accuracy': 0.4765625, 'macro_f1': 0.423672239457404, 'log_loss': 1.115755338768798, 'brier': 0.6559176617030762, 'n': 256}]



LightGBM per-tournament:


,year,accuracy,macro_f1,log_loss,brier,n
0,2010,0.390625,0.379823,1.20906,0.709206,64
1,2014,0.437500,0.362690,1.12804,0.661774,64
2,2018,0.500000,0.385166,1.10840,0.663394,64
3,2022,0.484375,0.425112,1.09534,0.629978,64



LightGBM pooled: [{'scope': 'pooled', 'accuracy': 0.453125, 'macro_f1': 0.3999106185434789, 'log_loss': 1.1352101268015564, 'brier': 0.6660882185830214, 'n': 256}]


In [7]:
# Side-by-side feature importances: Random Forest (Gini) vs XGBoost (gain)
xgb_model = train.fit_model(train.build_models()["xgboost"], df)
xgb_imp = pd.Series(xgb_model.feature_importances_,
                     index=features.FEATURE_COLUMNS).sort_values()
rf_imp = pd.Series(rf.feature_importances_,
                    index=features.FEATURE_COLUMNS).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_imp.plot(kind="barh", ax=axes[0], color="#4C72B0")
axes[0].set_title("Random Forest — Gini importance")
xgb_imp.plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("XGBoost — gain importance")
plt.tight_layout()
plt.savefig(FIG / "feature_importance_rf_vs_xgb.png", dpi=110)
plt.show()

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/4116448743.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### XGBoost hyperparameter sensitivity

A quick sweep over the key hyperparameters to check how sensitive the
model is on this small dataset. We vary one parameter at a time (holding the
others at their defaults) and measure pooled-backtest log loss.

In [8]:
def xgb_factory(**overrides):
    """Create an XGBClassifier with default params + overrides."""
    params = dict(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        objective="multi:softprob", num_class=3,
        random_state=config.RANDOM_SEED, n_jobs=-1,
        eval_metric="mlogloss",
    )
    params.update(overrides)
    return XGBClassifier(**params)

sweeps = {
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "n_estimators": [100, 200, 300, 500],
    "reg_lambda": [0.1, 0.5, 1.0, 3.0, 10.0],
}

sweep_results = []
for param_name, values in sweeps.items():
    for v in values:
        _, overall = evaluate.temporal_backtest(
            df, lambda v=v, p=param_name: xgb_factory(**{p: v}))
        if not overall.empty:
            sweep_results.append({
                "param": param_name, "value": v,
                "log_loss": overall.iloc[0]["log_loss"],
                "accuracy": overall.iloc[0]["accuracy"],
            })

sweep_df = pd.DataFrame(sweep_results)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, param_name in zip(axes.flat, sweeps.keys()):
    sub = sweep_df[sweep_df["param"] == param_name]
    ax.plot(sub["value"], sub["log_loss"], "o-", color="#DD8452", linewidth=2)
    ax.set_xlabel(param_name)
    ax.set_ylabel("log loss")
    ax.set_title(f"XGBoost sensitivity: {param_name}")
    best_row = sub.loc[sub["log_loss"].idxmin()]
    ax.axvline(best_row["value"], ls="--", color="grey", alpha=0.5)
    ax.annotate(f"best={best_row['value']}", xy=(best_row["value"], best_row["log_loss"]),
                fontsize=9, ha="left")
plt.tight_layout()
plt.savefig(FIG / "xgb_hyperparam_sensitivity.png", dpi=110)
plt.show()

# Show the best combination per param
print("Best value per parameter:")
for param_name in sweeps:
    sub = sweep_df[sweep_df["param"] == param_name]
    best = sub.loc[sub["log_loss"].idxmin()]
    print(f"  {param_name} = {best['value']}  (log_loss={best['log_loss']:.4f})")

Best value per parameter:
  max_depth = 2.0  (log_loss=1.0804)
  learning_rate = 0.01  (log_loss=1.0185)
  n_estimators = 100.0  (log_loss=1.0424)
  reg_lambda = 10.0  (log_loss=1.0819)


/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/3428310656.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Head-to-head comparison: all models per tournament

A visual comparison of log loss across the four backtest tournaments for every
model, making it easy to spot whether any model consistently dominates.

In [9]:
# Collect per-year results for all real models
model_names = [n for n in train.build_models() if "baseline" not in n]
all_per_year = []
for name in model_names:
    py, _ = evaluate.temporal_backtest(
        df, lambda n=name: train.build_models()[n])
    py["model"] = name
    all_per_year.append(py)
all_per_year_df = pd.concat(all_per_year, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ["log_loss", "accuracy"]):
    pivot = all_per_year_df.pivot(index="year", columns="model", values=metric)
    pivot.plot(kind="bar", ax=ax, width=0.75)
    ax.set_title(f"{metric} by tournament (temporal backtest)")
    ax.set_ylabel(metric)
    ax.legend(fontsize=8, loc="best")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(FIG / "all_models_per_year.png", dpi=110)
plt.show()

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/4048351288.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### XGBoost confusion matrix

Let's look at the XGBoost confusion matrix to see if its error pattern differs
from Random Forest's (e.g. does it predict draws more often?).

In [10]:
# XGBoost pooled confusion matrix
xgb_pt, xgb_pp = [], []
for ty in config.BACKTEST_YEARS:
    tr, te = df[df["year"] < ty], df[df["year"] == ty]
    if len(tr) and len(te):
        mdl = train.fit_model(train.build_models()["xgboost"], tr)
        xgb_pp.append(train.predict_proba(mdl, te))
        xgb_pt.append(te["target"].to_numpy())
xgb_cm = evaluate.confusion(np.concatenate(xgb_pt), np.concatenate(xgb_pp))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title(f"Random Forest (pooled backtest)")
sns.heatmap(xgb_cm, annot=True, fmt="d", cmap="Oranges", ax=axes[1])
axes[1].set_title("XGBoost (pooled backtest)")
plt.tight_layout()
plt.savefig(FIG / "confusion_rf_vs_xgb.png", dpi=110)
plt.show()

print("Random Forest confusion matrix:"); display(cm)
print("\nXGBoost confusion matrix:"); display(xgb_cm)

Random Forest confusion matrix:


/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/916088514.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,pred_team_a_win,pred_draw,pred_team_b_win
true_team_a_win,74,0,27
true_draw,31,0,26
true_team_b_win,29,1,68



XGBoost confusion matrix:


,pred_team_a_win,pred_draw,pred_team_b_win
true_team_a_win,61,14,26
true_draw,27,9,21
true_team_b_win,33,13,52


### Poisson goal model

We also fit a **Poisson regression** that predicts each team's expected goals
as a function of its attacking rate, the opponent's defensive rate, the Elo gap,
and whether it's a knockout match. This gives exact score predictions.

In [11]:
pois = poisson.PoissonGoalModel().fit(df)

# Poisson-derived outcome probabilities for the backtest
pois_pt, pois_pp = [], []
for ty in config.BACKTEST_YEARS:
    tr, te = df[df["year"] < ty], df[df["year"] == ty]
    if len(tr) and len(te):
        pm = poisson.PoissonGoalModel().fit(tr)
        pois_pp.append(pm.predict_proba(te)); pois_pt.append(te["target"].to_numpy())
pois_metrics = evaluate.compute_metrics(np.concatenate(pois_pt),
                                        np.concatenate(pois_pp))
print("Poisson model (pooled backtest):", pois_metrics)

Poisson model (pooled backtest): {'accuracy': 0.52734375, 'macro_f1': 0.3958333333333333, 'log_loss': 1.0131676021352998, 'brier': 0.6035847787561529, 'n': 256}


## 7. FIFA World Cup 2026 — Predictions

We train the best model (by backtest log loss — could now be XGBoost!) and the
Poisson model on **all** available data (1930–2022), snapshot the final
Elo/stats state, and predict the 72 group-stage matches. We also generate
XGBoost-specific predictions for comparison and run a Monte-Carlo tournament
simulation.

In [12]:
from scripts.build_fixture_2026 import GROUPS, HOSTS  # noqa: E402

best = str(best)  # ensure plain string (not numpy/pandas scalar)
clf, pois_model, hist, clf_name = predict.fit_full(best)
print(f"Using classifier: {clf_name}")

fixture = pd.read_csv(config.FIXTURE_2026_PATH)
_, unmatched = predict.resolve_team_ids(
    sorted(set(fixture["team_a"]) | set(fixture["team_b"])))
if unmatched:
    print(f"Debutant teams (no men's WC history, Elo={int(config.ELO_BASE)}):",
          unmatched)

preds = predict.predict_matches(fixture, clf, pois_model, hist, HOSTS)
cols = ["match_number", "group", "team_a", "team_b", "team_a_elo",
        "team_b_elo", "p_team_a_win", "p_draw", "p_team_b_win",
        "predicted_outcome_label", "predicted_score",
        "exp_goals_a", "exp_goals_b"]
preds[cols].to_csv(config.PROCESSED_DIR / "predictions_2026_groups.csv",
                   index=False)
preds[cols]

Using classifier: logistic_regression
Debutant teams (no men's WC history, Elo=1500): ['Jordan', 'Uzbekistan']


,match_number,group,team_a,team_b,team_a_elo,team_b_elo,p_team_a_win,p_draw,p_team_b_win,predicted_outcome_label,predicted_score,exp_goals_a,exp_goals_b
0,1,A,Mexico,Croatia,1530,1610,0.6723,0.2256,0.1022,Mexico,0-1,0.99,1.39
1,2,A,Mexico,Ivory Coast,1530,1491,0.7704,0.1496,0.0800,Mexico,1-1,1.21,1.22
2,3,A,Mexico,Saudi Arabia,1530,1341,0.8781,0.0730,0.0489,Mexico,1-0,1.60,0.90
3,4,A,Croatia,Ivory Coast,1610,1491,0.4528,0.2560,0.2912,Croatia,1-1,1.43,1.03
4,5,A,Croatia,Saudi Arabia,1610,1341,0.6198,0.1813,0.1989,Croatia,1-0,1.88,0.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Italy,Jordan,1657,1500,0.6577,0.2258,0.1165,Italy,1-0,1.53,0.99
68,69,L,Italy,Scotland,1657,1437,0.5729,0.2451,0.1820,Italy,1-0,1.69,0.84
69,70,L,Austria,Jordan,1510,1500,0.4750,0.2628,0.2622,Austria,1-1,1.28,1.29
70,71,L,Austria,Scotland,1510,1437,0.3390,0.2402,0.4207,Scotland,1-1,1.41,1.09


### XGBoost vs best-model predictions comparison

Let's also generate predictions with XGBoost specifically (if it wasn't already
the best model) and compare how the two classifiers disagree.

In [13]:
# Generate XGBoost-specific predictions (separate from whatever 'best' was)
xgb_clf, xgb_pois, xgb_hist, xgb_clf_name = predict.fit_full("xgboost")
print(f"XGBoost model trained on all history")

xgb_preds = predict.predict_matches(fixture, xgb_clf, xgb_pois, xgb_hist, HOSTS)
xgb_preds_cols = xgb_preds[cols].copy()
xgb_preds_cols.columns = [c + "_xgb" if c not in ["match_number", "group", "team_a", "team_b"]
                           else c for c in cols]

# Merge for side-by-side comparison
merged = preds[cols].merge(xgb_preds_cols,
                           on=["match_number", "group", "team_a", "team_b"],
                           suffixes=("", "_xgb"))

# Where do they disagree on the predicted winner?
disagree = merged[merged["predicted_outcome_label"] != merged["predicted_outcome_label_xgb"]]
print(f"\nDisagreements: {len(disagree)} of {len(merged)} matches")
if len(disagree) > 0:
    display(disagree[["team_a", "team_b",
                       "p_team_a_win", "p_draw", "p_team_b_win", "predicted_outcome_label",
                       "p_team_a_win_xgb", "p_draw_xgb", "p_team_b_win_xgb",
                       "predicted_outcome_label_xgb"]])
else:
    print("Both models agree on all match outcomes!")

XGBoost model trained on all history

Disagreements: 11 of 72 matches


,team_a,team_b,p_team_a_win,p_draw,p_team_b_win,predicted_outcome_label,p_team_a_win_xgb,p_draw_xgb,p_team_b_win_xgb,predicted_outcome_label_xgb
4,Croatia,Saudi Arabia,0.6198,0.1813,0.1989,Croatia,0.2587,0.3714,0.3699,Draw
13,Argentina,Korea Republic,0.5133,0.3782,0.1085,Argentina,0.2847,0.5605,0.1548,Draw
15,Poland,Korea Republic,0.3654,0.2388,0.3958,Korea Republic,0.6921,0.1064,0.2015,Poland
18,United States,Uruguay,0.4708,0.2490,0.2802,United States,0.0972,0.5750,0.3278,Draw
30,Brazil,Netherlands,0.1613,0.7805,0.0583,Draw,0.5757,0.3162,0.1081,Brazil
44,Germany,Cameroon,0.6477,0.1887,0.1636,Germany,0.3310,0.2746,0.3944,Cameroon
47,Iran,Cameroon,0.3321,0.2555,0.4124,Cameroon,0.3873,0.2448,0.3679,Iran
48,Portugal,Sweden,0.3047,0.2368,0.4585,Sweden,0.3864,0.3029,0.3107,Portugal
59,Morocco,Algeria,0.4127,0.2498,0.3375,Morocco,0.3429,0.2645,0.3926,Algeria
66,Italy,Austria,0.4418,0.4449,0.1133,Draw,0.6691,0.2205,0.1105,Italy


In [14]:
# Probability calibration: scatter plot of best-model vs XGBoost probabilities
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, label) in zip(axes, [("p_team_a_win", "P(A win)"),
                                    ("p_draw", "P(Draw)"),
                                    ("p_team_b_win", "P(B win)")]):
    ax.scatter(merged[col], merged[col + "_xgb"], alpha=0.5, s=30)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel(f"{clf_name} — {label}")
    ax.set_ylabel(f"XGBoost — {label}")
    ax.set_title(label)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect("equal")
plt.suptitle("Probability comparison: best model vs XGBoost", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG / "proba_comparison_best_vs_xgb.png", dpi=110, bbox_inches="tight")
plt.show()

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/336355254.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Monte-Carlo tournament simulation

In [15]:
sim = predict.simulate_tournament(GROUPS, clf, pois_model, hist, HOSTS,
                                  n_sims=3000)
sim.to_csv(config.PROCESSED_DIR / "tournament_simulation_2026.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_n = sim.head(20)
bars = ax.barh(top_n["team"][::-1], top_n["p_champion"][::-1], color="#4C72B0")
ax.set_xlabel("P(Champion)")
ax.set_title("Top 20 title contenders — FIFA World Cup 2026")
for b in bars:
    ax.text(b.get_width() + 0.002, b.get_y() + b.get_height() / 2,
            f"{b.get_width():.1%}", va="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "title_probabilities.png", dpi=110); plt.show()

sim.head(20)

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_9450/935055390.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "title_probabilities.png", dpi=110); plt.show()


,team,elo,p_advance,p_champion
0,Mexico,1529,0.9853,0.1950
1,France,1775,0.9593,0.1080
2,Brazil,1724,0.9570,0.1050
3,Argentina,1713,0.9633,0.0850
4,United States,1445,0.9367,0.0633
5,Spain,1631,0.9787,0.0630
6,Netherlands,1804,0.9337,0.0617
7,Italy,1656,0.9223,0.0603
8,England,1640,0.8943,0.0483
9,Belgium,1625,0.9340,0.0410


## Interpretation & caveats

These predictions are only as good as the *historical World Cup data* allows:

* **No data between tournaments.** Elo ratings only update every 4 years at the
  World Cup — real team strength changes in between (qualifiers, friendlies,
  continental cups). External Elo/FIFA ranking data would substantially help.
* **Squad and player changes** are invisible to a results-only model. 2022
  champions Argentina may have a weaker/stronger squad in 2026; the model cannot
  know this.
* **Defunct teams.** West Germany, USSR, Yugoslavia hold high Elo that modern
  Germany, Russia, Serbia do not inherit. Germany's current rating is based only
  on matches played under that exact name in the dataset.
* **48-team expansion.** 2026 has a new format (12 groups of 4, 32 R-of-32).
  This is structurally different from anything the model trained on; the
  tournament-simulation bracket rules are an approximation.
* **Draw class remains hard.** No model reliably predicts when a match will be a
  draw; probabilities ~20–25% are the best that pure-results models achieve.

For a production forecaster, the next-highest-value improvements would be:
1. Incorporate international matches outside the World Cup (doubles the data).
2. Use the official FIFA/Elo ranking (much finer-grained strength estimates).
3. Add squad-level data (injuries, market value, age profile).
4. Use betting-market implied probabilities as calibration targets.